# Analyze the EEG data using MNE-Python for us

In [ ]:
import os
import numpy as np
import mne
import matplotlib.pyplot as plt

from pathlib import Path

In [ ]:
base = Path("/Users/trinity/Documents/GitHub/eeg-hackXplore")

fif_file_path = base / "measurement data-the amplitude" / "fif-files" / "med2_27_06_2026_10_30_13_raw.fif"

assert fif_file_path.exists(), "File not found"

raw = mne.io.read_raw_fif(fif_file_path, preload=True)

In [ ]:
# signal preprocessing - only relevant for raw data
raw = raw.notch_filter(freqs=[50], picks="eeg")
raw = raw.filter(l_freq=0.1, h_freq=40, picks="eeg")
# raw.drop_channels('Cz')  # drop channels if needed

# whiten the data

In [ ]:
# 1. Künstliche Events alle 2 Sekunden erstellen (ersetzt find_events)
# Das simuliert den kontinuierlichen Informationsfluss an den OP-Roboter
events = mne.make_fixed_length_events(raw, id=1, duration=2.0)

# 2. Epochen aus diesen Events generieren (0 bis 2 Sekunden pro Epoche)
epochs = mne.Epochs(raw, events, event_id=1, tmin=0, tmax=2.0, 
                    baseline=None, preload=True, verbose=False)

print(f"Erfolgreich {len(epochs)} Epochen (OP-Zeitfenster) erstellt!")

In [ ]:
# Hilfsfunktion, um die Band-Power (Energie in einem Frequenzbereich) zu berechnen
def get_band_power(epochs, fmin, fmax, channels):
    # Berechnet die Power Spectral Density (PSD) mittels Welch-Methode
    spectrum = epochs.compute_psd(method='welch', fmin=fmin, fmax=fmax, 
                                  picks=channels, verbose=False)
    # Extrahiere die Daten: Form (Epochen, Kanäle, Frequenzen)
    data = spectrum.get_data()
    # Mittelwert über Frequenzen (axis=2) und Kanäle (axis=1) -> 1 Wert pro Epoche
    return data.mean(axis=(1, 2))

# 1. Feature: Kognitiver Workload (Theta an der Stirn)
# Steigt bei Fehlern oder komplexen OP-Schritten
workload_feature = get_band_power(epochs, fmin=4, fmax=8, channels=['Fz'])

# 2. Feature: Motorische Vorbereitung (Beta zentral)
# Sinkt (Desynchronisation), wenn der Chirurg eine Bewegung plant/ausführt
motor_feature = get_band_power(epochs, fmin=13, fmax=30, channels=['C3', 'C4'])

# 3. Feature: Visueller Fokus (Alpha am Hinterkopf)
# Gibt an, wie stark der Chirurg auf das Mikroskop/Display fokussiert ist
focus_feature = get_band_power(epochs, fmin=8, fmax=12, channels=['Pz', 'Oz'])

# Alles in einen sauberen Pandas DataFrame packen (X-Daten für Machine Learning)
df_features = pd.DataFrame({
    'Workload_Theta': workload_feature,
    'Motorik_Beta': motor_feature,
    'Fokus_Alpha': focus_feature
})

display(df_features.head())

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# --- MOCKUP FÜR DEN HACKATHON ---
# Wir simulieren 3 OP-Phasen: 0=Beobachtung, 1=Präzisionsschnitt, 2=Kritische Phase
# In der Realität kämen diese Labels aus einem getaggten OP-Video.
np.random.seed(42)
y_labels = np.random.randint(0, 3, size=len(df_features))
# --------------------------------

# 1. Daten in Trainings- und Testset aufteilen (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(df_features, y_labels, test_size=0.2, random_state=42)

# 2. Den Classifier initialisieren und trainieren
clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
clf.fit(X_train, y_train)

# 3. Vorhersagen treffen
y_pred = clf.predict(X_test)

# 4. Auswertung anzeigen
print("Klassifizierungs-Report (Simulierte OP-Phasen):")
print(classification_report(y_test, y_pred, target_names=['Beobachtung', 'Präzision', 'Kritisch']))